# SI4006 · Entrega M1 — Lector de fichas de crédito

**Qué hace este sistema.** Le damos a un modelo de lenguaje la descripción escrita de una
persona que pide un crédito —su edad, en qué trabaja, cuánto gana, cómo ha pagado antes, cuánto
pide y para qué— y responde qué tan probable es que deje de pagar durante el primer año. Está
pensado para el analista que hoy lee esas fichas una por una: el modelo se las ordena de mayor
a menor riesgo.

| | |
| :--- | :--- |
| **Entrada** | La ficha de un solicitante, redactada en español (~1.200 caracteres) |
| **Salida** | `incumple` / `paga` — ¿deja de pagar en sus primeros 12 meses? |
| **Familia** | **Encoder** (BERT) |
| **Modelo base** | `dccuchile/bert-base-spanish-wwm-cased` (BETO, 110M) |
| **Adaptación** | **LoRA** — se entrena menos del 2% de los parámetros |
| **Métrica principal** | **PR-AUC** (precisión media) |

El notebook corre de principio a fin y tiene dos partes:

- **Parte A — De la tabla sucia al texto.** `perfiles.csv` tiene 252.523 filas, 72 columnas y no
  es un dataset de texto: trae ids repetidos, cuatro formatos de fecha, centinelas disfrazados
  de números, la misma ciudad escrita de 95 maneras y columnas que no se pueden usar por
  razones éticas o porque filtran el futuro. Aquí se limpia, se define la etiqueta y se redacta
  una ficha por cliente.
- **Parte B — Fine-tuning con LoRA.** Se afina BETO sobre esas fichas y se compara contra tres
  baselines sobre el mismo conjunto de test.

> **Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.
> Tiempo total aproximado: 5 min la Parte A (CPU) + 20 min la Parte B (GPU).

## 0 · Setup

Colab ya trae `torch` y `transformers`; instalamos sólo lo que falta.

In [2]:
%pip install -q peft datasets evaluate accelerate
# Colab trae un torchao viejo que choca con peft en get_peft_model. No lo usamos aquí.
%pip uninstall -y -q torchao
print('Listo.')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Listo.


In [3]:
import os, re, json, random, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch, transformers, peft

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA); torch.cuda.manual_seed_all(SEMILLA)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pd.set_option('display.width', 200)

print('transformers', transformers.__version__, '| peft', peft.__version__)
print('torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('\n⚠️  Estás en CPU. La Parte A corre igual; la Parte B entrena lentísimo.')

c:\Users\camil\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 5.14.1 | peft 0.21.0
torch 2.10.0+cpu | device: cpu

⚠️  Estás en CPU. La Parte A corre igual; la Parte B entrena lentísimo.


---
# Parte A · De la tabla sucia al dataset de texto

## 1 · Cargar `perfiles.csv`

En Colab: suban el archivo cuando lo pida la celda, o monten Drive si ya está allí.

In [4]:
RUTA_CSV = Path('perfiles.csv')

if not RUTA_CSV.exists():
    try:
        from google.colab import files
        print('Suban perfiles.csv (pesa ~140 MB, puede tardar):')
        subida = files.upload()
        RUTA_CSV = Path(next(iter(subida)))
    except ImportError:
        raise FileNotFoundError(
            'No encuentro perfiles.csv. Póngalo junto al notebook o ajuste RUTA_CSV.')

crudo = pd.read_csv(RUTA_CSV, low_memory=False)
print(crudo.shape)
crudo.head(3)

(252523, 72)


,id_usuario,fecha_vinculacion,ultima_actualizacion,origen_registro,id_asesor,version_politica,edad,sexo,ciudad,region,...,meses_observados,tiempo_evento,evento,clv_estimado,propension_churn,ruido_1,ruido_2,ruido_3,ruido_4,ruido_5
0,U-306679,2013-07-02,2014-04-23,digital,198,P-2024,65,F,Medellín,Andina,...,33,33,0,24514000.0,0.01,-1.7077,-1.0515,-0.1648,-1.4895,-0.5349
1,U-210735,2012-05-09,2013-05-19,sucursal,165,P-2024,33,M,Popayán,Pacífica,...,35,35,0,6327000.0,0.27,-1.0899,-0.9098,-1.0491,-0.2143,-0.9144
2,U-052357,15-Feb-2011,2013-05-07,aliado,472,P-2024,51,M,BOGOTÁ,Andina,...,31,31,0,9941000.0,0.01,-0.6110,0.8278,0.5929,0.0568,0.8225


## 2 · Inventario de trampas

Antes de tocar nada, medimos el daño. Cada número de abajo es una decisión que hay que tomar
explícitamente; ninguna la resuelve un `dropna()`.

In [5]:
def contar(nombre, condicion):
    n = int(condicion.sum())
    return {'problema': nombre, 'filas': n, '% del archivo': round(100 * n / len(crudo), 2)}

num = lambda c: pd.to_numeric(crudo[c], errors='coerce')

diagnostico = pd.DataFrame([
    contar('id_usuario repetido',              crudo.id_usuario.duplicated()),
    contar('edad imposible (<18 o >95)',       (num('edad') < 18) | (num('edad') > 95)),
    contar('score_credito centinela -999',     num('score_credito') == -999),
    contar('max_dias_mora centinela 999',      num('max_dias_mora_historico') == 999),
    contar('num_entidades centinela 99',       num('num_entidades_reportadas') == 99),
    contar('dependientes centinela -1',        num('dependientes') == -1),
    contar('sesiones_app centinela -1',        num('sesiones_app_mes') == -1),
    contar('ingreso en miles declarado',       crudo.unidad_ingreso == 'miles_COP'),
    contar('ingreso en miles SIN unidad',      (num('ingreso_reportado') < 50_000) & crudo.unidad_ingreso.isna()),
    contar('patrimonio en USD',                crudo.moneda == 'USD'),
    contar('coordenadas en (0,0)',             (num('latitud') == 0) & (num('longitud') == 0)),
    contar('coordenadas invertidas',           (num('longitud') > 0) | (num('latitud') < -5)),
    contar('score_bureau_externo ausente',     crudo.score_bureau_externo.isna()),
    contar('nps_declarado ausente',            crudo.nps_declarado.isna()),
])
display(diagnostico)

print('ciudades distintas en el archivo    :', crudo.ciudad.nunique(), '(reales: 22)')
print('codificaciones de propiedad_vivienda:', sorted(crudo.propiedad_vivienda.dropna().astype(str).unique()))
print('codificaciones de uso_digital_alto  :', sorted(crudo.uso_digital_alto.dropna().astype(str).unique()))
print('valores de version_politica         :', crudo.version_politica.unique(), '<- constante')
print('ejemplos de utilizacion_tdc         :', crudo.utilizacion_tdc.dropna().astype(str).unique()[:6])

,problema,filas,% del archivo
0,id_usuario repetido,2623,1.04
1,edad imposible (<18 o >95),445,0.18
2,score_credito centinela -999,8693,3.44
3,max_dias_mora centinela 999,8799,3.48
4,num_entidades centinela 99,9023,3.57
5,dependientes centinela -1,9028,3.58
6,sesiones_app centinela -1,8759,3.47
7,ingreso en miles declarado,7543,2.99
8,ingreso en miles SIN unidad,7027,2.78
9,patrimonio en USD,9210,3.65


ciudades distintas en el archivo    : 95 (reales: 22)
codificaciones de propiedad_vivienda: ['0', '1', 'False', 'N', 'No', 'S', 'Si', 'True']
codificaciones de uso_digital_alto  : ['0', '1', 'False', 'N', 'No', 'S', 'Si', 'True']
valores de version_politica         : <ArrowStringArray>
['P-2024']
Length: 1, dtype: str <- constante
ejemplos de utilizacion_tdc         : <ArrowStringArray>
['0.7594', '0.4088', '38,1%', '0.5812', '0.1311', '41,9%']
Length: 6, dtype: str


`fecha_vinculacion` mezcla tres plantillas y dos fechas centinela. Pasarle esto a
`pd.to_datetime` de un tirón produce fechas silenciosamente incorrectas: el `03/07` se lee como
3 de julio o como 7 de marzo según el humor del parser.

In [6]:
s = crudo.fecha_vinculacion.dropna().astype(str)
formatos = {
    'ISO aaaa-mm-dd': int(s.str.match(r'^\d{4}-\d{2}-\d{2}$').sum()),
    'dd/mm/aaaa':     int(s.str.match(r'^\d{2}/\d{2}/\d{4}$').sum()),
    'dd-Mon-aaaa':    int(s.str.match(r'^\d{2}-[A-Za-z]{3}-\d{4}$').sum()),
}
formatos['otro'] = len(s) - sum(formatos.values())
print(formatos)
print('centinelas:', dict(s[s.isin(['1900-01-01', '2031-12-31'])].value_counts()))
print('ejemplos dd-Mon:', s[s.str.match(r'^\d{2}-[A-Za-z]{3}-\d{4}$')].head(4).tolist())

{'ISO aaaa-mm-dd': 138924, 'dd/mm/aaaa': 88131, 'dd-Mon-aaaa': 25468, 'otro': 0}
centinelas: {'1900-01-01': np.int64(339), '2031-12-31': np.int64(330)}
ejemplos dd-Mon: ['15-Feb-2011', '10-Jun-2013', '30-Jan-2013', '06-Oct-2023']


## 3 · Normalizadores

Una función por trampa. Se prueban abajo antes de aplicarlas a las 252.523 filas.

In [7]:
# --- valores centinela: un número imposible que en realidad significa "sin dato"
CENTINELAS = {
    'score_credito': [-999],
    'max_dias_mora_historico': [999],
    'num_entidades_reportadas': [99],
    'dependientes': [-1],
    'sesiones_app_mes': [-1],
}

# --- las 22 ciudades reales del archivo
CIUDADES_CANONICAS = [
    'Bogotá', 'Medellín', 'Cali', 'Barranquilla', 'Cartagena', 'Bucaramanga',
    'Cúcuta', 'Pereira', 'Santa Marta', 'Manizales', 'Villavicencio', 'Ibagué',
    'Pasto', 'Neiva', 'Armenia', 'Sincelejo', 'Valledupar', 'Montería',
    'Tunja', 'Popayán', 'Riohacha', 'Quibdó',
]

MESES_EN = {'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04', 'may': '05', 'jun': '06',
            'jul': '07', 'aug': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'}

VERDADEROS = {'true', 'si', 'sí', 's', '1', 'y', 'yes'}
FALSOS     = {'false', 'no', 'n', '0'}
# textos que significan "sin dato" pero llegaron como si fueran una categoría más
NULOS_TEXTO = {'', '-', 'n/a', 'na', 'sin informacion', 'sin información', 'none', 'nan', 'null'}

TRM_SUPUESTA = 4000.0   # COP/USD: supuesto explícito, el archivo no trae fecha de conversión


def sin_tildes(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto)
                   if unicodedata.category(c) != 'Mn')


def normalizar_categoria(valor):
    if pd.isna(valor):
        return np.nan
    limpio = str(valor).strip()
    return np.nan if sin_tildes(limpio).lower() in NULOS_TEXTO else limpio


def normalizar_booleano(valor):
    if pd.isna(valor):
        return np.nan
    limpio = sin_tildes(str(valor).strip()).lower()
    if limpio in VERDADEROS: return 1.0
    if limpio in FALSOS:     return 0.0
    return np.nan


def normalizar_ciudad(valor):
    if pd.isna(valor):
        return np.nan
    limpio = re.sub(r'\s+', ' ', str(valor)).strip()
    limpio = re.sub(r'\bD\.?\s?C\.?\b', '', limpio, flags=re.IGNORECASE).strip()
    clave = sin_tildes(limpio).lower()
    for canonica in CIUDADES_CANONICAS:
        if sin_tildes(canonica).lower() == clave:
            return canonica
    return limpio.title() if limpio else np.nan


def normalizar_fecha(valor):
    if pd.isna(valor):
        return pd.NaT
    texto = str(valor).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', texto):
        fecha = pd.to_datetime(texto, format='%Y-%m-%d', errors='coerce')
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', texto):
        fecha = pd.to_datetime(texto, format='%d/%m/%Y', errors='coerce')
    elif re.match(r'^\d{2}-[A-Za-z]{3}-\d{4}$', texto):
        dia, mes, anio = texto.split('-')
        mes_num = MESES_EN.get(mes.lower())
        fecha = pd.to_datetime(f'{anio}-{mes_num}-{dia}', errors='coerce') if mes_num else pd.NaT
    else:
        fecha = pd.to_datetime(texto, errors='coerce', dayfirst=True)
    if pd.isna(fecha) or fecha.year <= 1901 or fecha.year >= 2027:
        return pd.NaT     # 1900-01-01 y las fechas futuras son centinelas
    return fecha


def normalizar_utilizacion(valor):
    """35% viene como '35,0%' (coma decimal, base 100) y también como 0.35."""
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip()
    porcentaje = texto.endswith('%')
    try:
        numero = float(texto.rstrip('%').replace(',', '.'))
    except ValueError:
        return np.nan
    return numero / 100.0 if (porcentaje or numero > 1.5) else numero


def tendencia_saldos(valor):
    """'a|b|c|d|e|f' -> último trimestre contra el primero."""
    if pd.isna(valor):
        return np.nan
    try:
        saldos = [float(x) for x in str(valor).split('|')]
    except ValueError:
        return np.nan
    if len(saldos) < 6:
        return np.nan
    inicio, fin = np.mean(saldos[:3]), np.mean(saldos[3:])
    if inicio <= 0:
        return np.nan
    cambio = (fin - inicio) / inicio
    return 'creciente' if cambio > 0.10 else ('decreciente' if cambio < -0.10 else 'estable')


def contar_productos(valor):
    if pd.isna(valor):
        return np.nan
    try:
        return float(len(json.loads(valor)))
    except (json.JSONDecodeError, TypeError):
        return np.nan


# --- comprobación rápida
print(normalizar_ciudad('  BOGOTÁ D.C. '), '|', normalizar_ciudad('medellin'))
print(normalizar_booleano('S'), normalizar_booleano('No'), normalizar_booleano('1'))
print(normalizar_utilizacion('32,1%'), normalizar_utilizacion(0.7594))
print(normalizar_fecha('15-Feb-2011'), '|', normalizar_fecha('13/07/2019'), '|', normalizar_fecha('1900-01-01'))
print(tendencia_saldos('100|100|100|200|200|200'), '|', contar_productos('["CDT", "Crédito Consumo"]'))

Bogotá . | Medellín
1.0 0.0 1.0
0.321 0.7594
2011-02-15 00:00:00 | 2019-07-13 00:00:00 | NaT
creciente | 2.0


## 4 · Limpieza

Dos decisiones que no son mecánicas y conviene justificar:

- **Ingreso sin unidad.** Cuando falta `unidad_ingreso` y el valor es menor a 50.000, se asume
  que venía en miles: un ingreso mensual de 3.660 pesos no existe. Queda marcado en
  `ingreso_unidad_imputada` para poder auditarlo.
- **Buró externo ausente.** No se imputa. Falta *más cuando el cliente es peor* (MNAR):
  rellenarlo con la mediana borra justamente la señal. Se deja nulo y en la ficha se escribirá
  "sin dato", que es información real que el modelo puede leer.

In [8]:
def limpiar(df):
    d = df.copy()

    # Ids repetidos: nos quedamos con el refresco más reciente de cada cliente, para
    # que el mismo caso no aparezca después en train y en test a la vez.
    d['ultima_actualizacion'] = d['ultima_actualizacion'].map(normalizar_fecha)
    d = (d.sort_values('ultima_actualizacion', na_position='first')
           .drop_duplicates(subset='id_usuario', keep='last')
           .reset_index(drop=True))

    d['fecha_vinculacion'] = d['fecha_vinculacion'].map(normalizar_fecha)
    d['anios_vinculado'] = (pd.Timestamp('2026-01-01') - d['fecha_vinculacion']).dt.days / 365.25
    d.loc[d['anios_vinculado'] < 0, 'anios_vinculado'] = np.nan

    for columna, valores in CENTINELAS.items():
        d[columna] = pd.to_numeric(d[columna], errors='coerce').replace(valores, np.nan)

    d['edad'] = pd.to_numeric(d['edad'], errors='coerce')
    d.loc[(d['edad'] < 18) | (d['edad'] > 95), 'edad'] = np.nan

    for columna in ['tipo_empleo', 'receptividad_tecnologica', 'canal_preferido',
                    'nivel_educativo', 'estado_civil', 'tipo_vivienda', 'region',
                    'profesion', 'motivo_credito', 'descripcion_ocupacion', 'origen_registro']:
        d[columna] = d[columna].map(normalizar_categoria)

    d['ciudad'] = d['ciudad'].map(normalizar_ciudad)

    for columna in ['propiedad_vivienda', 'uso_digital_alto', 'tiene_garantia_disponible']:
        d[columna] = d[columna].map(normalizar_booleano)

    d['utilizacion_tdc']  = d['utilizacion_tdc'].map(normalizar_utilizacion)
    d['tendencia_saldos'] = d['saldos_ultimos_6m'].map(tendencia_saldos)
    d['num_productos']    = d['productos_actuales_json'].map(contar_productos)

    # Ingreso: 6% viene en miles y en la mitad de esos casos falta la unidad.
    d['ingreso_reportado'] = pd.to_numeric(d['ingreso_reportado'], errors='coerce')
    en_miles         = d['unidad_ingreso'].astype(str).str.strip() == 'miles_COP'
    unidad_ausente   = d['unidad_ingreso'].isna()
    magnitud_absurda = d['ingreso_reportado'] < 50_000
    d['ingreso_unidad_imputada'] = (unidad_ausente & magnitud_absurda).astype(int)
    d['ingreso_mensual_cop'] = d['ingreso_reportado'].where(
        ~(en_miles | (unidad_ausente & magnitud_absurda)), d['ingreso_reportado'] * 1000)

    # Patrimonio: 4% en dólares.
    d['patrimonio_declarado'] = pd.to_numeric(d['patrimonio_declarado'], errors='coerce')
    es_usd = d['moneda'].astype(str).str.strip().str.upper() == 'USD'
    d['patrimonio_cop'] = d['patrimonio_declarado'].where(
        ~es_usd, d['patrimonio_declarado'] * TRM_SUPUESTA)

    for columna in ['estrato', 'anios_ciudad_actual', 'volatilidad_ingreso',
                    'deuda_total_actual', 'ratio_deuda_ingreso', 'cuota_mensual_actual',
                    'score_bureau_externo', 'antiguedad_financiera', 'num_creditos_activos',
                    'transacciones_digitales_mes', 'gasto_mensual_promedio',
                    'pct_gasto_esencial', 'moras_ultimos_24m', 'meses_desde_ultima_mora',
                    'consultas_centrales_6m', 'monto_solicitado', 'plazo_solicitado',
                    'meses_observados', 'tiempo_evento', 'evento']:
        d[columna] = pd.to_numeric(d[columna], errors='coerce')

    return d


limpio = limpiar(crudo)
print(f'{len(crudo):,} filas crudas  ->  {len(limpio):,} tras deduplicar por id_usuario')

252,523 filas crudas  ->  249,900 tras deduplicar por id_usuario


In [9]:
comparacion = pd.DataFrame({
    'antes':   [crudo.ciudad.nunique(),
                crudo.propiedad_vivienda.dropna().nunique(),
                crudo.tipo_empleo.dropna().nunique(),
                int((pd.to_numeric(crudo.score_credito, errors='coerce') == -999).sum()),
                str(crudo.utilizacion_tdc.dropna().iloc[0])],
    'después': [limpio.ciudad.nunique(),
                int(limpio.propiedad_vivienda.dropna().nunique()),
                limpio.tipo_empleo.dropna().nunique(),
                int((limpio.score_credito == -999).sum()),
                round(float(limpio.utilizacion_tdc.dropna().iloc[0]), 4)],
}, index=['ciudades', 'códigos de propiedad_vivienda', 'niveles de tipo_empleo',
          'scores con centinela -999', 'ejemplo de utilizacion_tdc'])
display(comparacion)

print('Ingreso mensual en COP tras normalizar unidades:')
print(limpio.ingreso_mensual_cop.describe(percentiles=[.01, .5, .99]).round(0).to_string())
print(f'\nfilas con la unidad de ingreso imputada: {int(limpio.ingreso_unidad_imputada.sum()):,}')

,antes,después
ciudades,95,44.0000
códigos de propiedad_vivienda,8,2.0000
niveles de tipo_empleo,8,5.0000
scores con centinela -999,8693,0.0000
ejemplo de utilizacion_tdc,0.7594,0.2691


Ingreso mensual en COP tras normalizar unidades:
count       230410.0
mean       2521415.0
std        2515806.0
min          52125.0
1%          409000.0
50%        1850000.0
99%       11252910.0
max      160320000.0

filas con la unidad de ingreso imputada: 6,954


## 5 · La etiqueta: por qué `evento` solo no sirve

`evento = 0` **no significa "buen cliente"**. Puede significar que pagó bien durante 48 meses o
que sólo se le observó cuatro. Mezclar ambos casos le enseña al modelo que un perfil riesgoso
con poco seguimiento es seguro.

La corrección es fijar un horizonte común: nos quedamos con los clientes observados **al menos
12 meses** y preguntamos *¿incumplió dentro de esos 12 meses?*. Los de menos seguimiento **se
descartan** — de ellos no conocemos el desenlace, y descartarlos es más honesto que llamarlos
buenos.

In [10]:
HORIZONTE_MESES = 12

sin_seguimiento = limpio[(limpio.evento == 0) & (limpio.meses_observados < HORIZONTE_MESES)]
print('Si ignoráramos meses_observados, esto contaríamos como "buenos clientes":')
print(f'  {len(sin_seguimiento):,} perfiles con evento=0 pero menos de 12 meses de observación')
print(f'  ({100 * len(sin_seguimiento) / len(limpio):.1f}% del archivo etiquetado al revés)\n')

etiquetado = limpio[limpio.meses_observados >= HORIZONTE_MESES].copy()
etiquetado['y'] = ((etiquetado.evento == 1) &
                   (etiquetado.tiempo_evento <= HORIZONTE_MESES)).astype(int)

print(f'clientes con horizonte completo  : {len(etiquetado):,}')
print(f'tasa de incumplimiento a 12 meses: {etiquetado.y.mean():.4f}\n')
print('Tasa por tipo de empleo (comprobación de que la etiqueta tiene sentido):')
print(etiquetado.groupby('tipo_empleo').y.agg(['mean', 'size']).round(4).to_string())

Si ignoráramos meses_observados, esto contaríamos como "buenos clientes":
  43,535 perfiles con evento=0 pero menos de 12 meses de observación
  (17.4% del archivo etiquetado al revés)

clientes con horizonte completo  : 205,558
tasa de incumplimiento a 12 meses: 0.0371

Tasa por tipo de empleo (comprobación de que la etiqueta tiene sentido):
                        mean   size
tipo_empleo                        
Desempleado           0.0470   7426
Formal                0.0217  93427
Independiente formal  0.0258  39009
Informal              0.0706  56777
Pensionado            0.0078   2695


## 6 · Qué columnas quedan fuera, y por qué

De las 72 columnas, 27 no entran al modelo. No es poda por conveniencia: cada exclusión tiene
un motivo que se sostiene frente a un auditor.

In [11]:
COLUMNAS_EXCLUIDAS = {
    'sexo': 'variable protegida: no se usa como predictor',
    'castigos_previos': 'fuga de información (contaminada con el futuro)',
    'segmento_mercado': 'es la partición del simulador, no un hallazgo propio',
    'clv_estimado': 'se calcula después del desenlace',
    'propension_churn': 'se calcula después del desenlace',
    'score_politica_anterior': 'codifica la decisión de la política vieja (sesgo de selección)',
    'version_politica': 'constante, no aporta varianza',
    'id_asesor': '480 niveles; señal real pero es un proxy operativo, no del cliente',
    'latitud': '1.2% invertidas y 2% en (0,0); redundante con ciudad/región',
    'longitud': '1.2% invertidas y 2% en (0,0); redundante con ciudad/región',
    'ingresos_anuales': 'el multiplicador depende del tipo de empleo; se usa el ingreso mensual',
    'nps_declarado': '62% ausente y no es una señal de riesgo',
    'email_dominio': 'proxy socioeconómico sin relación causal con el pago',
    'propiedad_vivienda': 'redundante con tipo_vivienda (nulas en las mismas filas)',
    'codigo_ciiu': '220 niveles; redundante con profesion y descripcion_ocupacion',
    'receptividad_tecnologica': 'actitud comercial, no comportamiento de pago',
    'apertura_ofertas_pct': 'métrica de campaña, no de riesgo',
    'respuesta_campanas_previas': 'métrica de campaña, no de riesgo',
    'sesiones_app_mes': 'uso de canal; redundante con transacciones_digitales_mes',
    'ruido_1': 'aleatoria pura', 'ruido_2': 'aleatoria pura', 'ruido_3': 'aleatoria pura',
    'ruido_4': 'aleatoria pura', 'ruido_5': 'aleatoria pura',
    'evento': 'componente de la etiqueta',
    'tiempo_evento': 'componente de la etiqueta',
    'meses_observados': 'componente de la etiqueta',
}
display(pd.DataFrame(sorted(COLUMNAS_EXCLUIDAS.items()), columns=['columna', 'motivo']))
print(f'{len(COLUMNAS_EXCLUIDAS)} columnas excluidas de 72.')

,columna,motivo
0,apertura_ofertas_pct,"métrica de campaña, no de riesgo"
1,castigos_previos,fuga de información (contaminada con el futuro)
2,clv_estimado,se calcula después del desenlace
3,codigo_ciiu,220 niveles; redundante con profesion y descri...
4,email_dominio,proxy socioeconómico sin relación causal con e...
5,evento,componente de la etiqueta
6,id_asesor,480 niveles; señal real pero es un proxy opera...
7,ingresos_anuales,el multiplicador depende del tipo de empleo; s...
8,latitud,"1.2% invertidas y 2% en (0,0); redundante con ..."
9,longitud,"1.2% invertidas y 2% en (0,0); redundante con ..."


27 columnas excluidas de 72.


Las cinco `ruido_*` son la prueba del enunciado: son aleatorias puras, y están ahí para
separar a quien selecciona variables con criterio de quien lanza todas las columnas al modelo.
Se puede verificar.

In [12]:
for c in ['ruido_1', 'ruido_2', 'ruido_3', 'ruido_4', 'ruido_5']:
    r = np.corrcoef(pd.to_numeric(etiquetado[c], errors='coerce').fillna(0), etiquetado.y)[0, 1]
    print(f'{c}: correlación con la etiqueta = {r:+.4f}')

ruido_1: correlación con la etiqueta = +0.0033
ruido_2: correlación con la etiqueta = -0.0012
ruido_3: correlación con la etiqueta = -0.0012
ruido_4: correlación con la etiqueta = -0.0002
ruido_5: correlación con la etiqueta = +0.0010


## 7 · Splits y submuestreo

Split estratificado 70/15/15 con semilla fija. Los ids ya son únicos, así que ningún cliente
cruza de un conjunto a otro.

En **train** se submuestrean los negativos a 3 por cada positivo: con 3,7% de positivos el
modelo aprende a decir "paga" siempre. **Validación y test se dejan con la prevalencia real**,
porque es contra esa población que se va a usar el modelo; reequilibrar la evaluación subiría
el F1 sin que el modelo detectara un solo cliente más.

Partimos **antes** de redactar las fichas: así se redactan ~61.000 y no 205.000.

In [13]:
N_TRAIN, N_EVAL = 24_000, 20_000
RATIO_NEGATIVOS = 3

rng = np.random.default_rng(SEMILLA)
partes = {'train': [], 'validation': [], 'test': []}
for _, grupo in etiquetado.groupby('y'):
    orden = rng.permutation(len(grupo))
    n_val, n_test = int(len(grupo) * 0.15), int(len(grupo) * 0.15)
    partes['validation'].append(grupo.iloc[orden[:n_val]])
    partes['test'].append(grupo.iloc[orden[n_val:n_val + n_test]])
    partes['train'].append(grupo.iloc[orden[n_val + n_test:]])
partes = {k: pd.concat(v).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
          for k, v in partes.items()}

# train: todos los incumplimientos + 3 negativos por cada uno
positivos = partes['train'][partes['train'].y == 1]
negativos = partes['train'][partes['train'].y == 0].sample(
    n=min((partes['train'].y == 0).sum(), len(positivos) * RATIO_NEGATIVOS),
    random_state=SEMILLA)
partes['train'] = (pd.concat([positivos, negativos])
                     .sample(frac=1, random_state=SEMILLA).reset_index(drop=True).head(N_TRAIN))
partes['validation'] = partes['validation'].head(N_EVAL)
partes['test']       = partes['test'].head(N_EVAL)

display(pd.DataFrame([{'split': k, 'n': len(v), 'incumplen': int(v.y.sum()),
                       'tasa': round(float(v.y.mean()), 4)} for k, v in partes.items()]))

ids = {k: set(v.id_usuario) for k, v in partes.items()}
print('clientes compartidos entre splits:',
      len(ids['train'] & ids['validation']), len(ids['train'] & ids['test']),
      len(ids['validation'] & ids['test']))

,split,n,incumplen,tasa
0,train,21352,5338,0.2500
1,validation,20000,729,0.0365
2,test,20000,729,0.0365


clientes compartidos entre splits: 0 0 0


## 8 · La ficha: de fila a texto

El transformer no lee columnas, lee prosa. Cada cliente se escribe como lo describiría un
analista, en seis bloques: perfil, ocupación, situación crediticia, comportamiento de pago,
relación con la entidad y solicitud.

Tres criterios en la redacción:

- **Los nulos se escriben, no se borran.** "score de buró externo sin dato" es una frase que el
  modelo puede aprender a leer como señal de riesgo. Un cero imputado, no.
- **Los montos se redondean a millones.** Se pierde precisión numérica a cambio de tokens que
  el modelo ya vio en español. Es un costo consciente y está en las limitaciones.
- **Se conserva el texto libre original.** `descripcion_ocupacion` y `motivo_credito` entran tal
  cual: son las únicas columnas que ya eran lenguaje natural.

In [14]:
def _millones(v):
    if pd.isna(v): return 'sin dato'
    if abs(v) >= 1_000_000: return f'{v / 1_000_000:.1f} millones de pesos'
    if abs(v) >= 1_000:     return f'{v / 1_000:.0f} mil pesos'
    return f'{v:.0f} pesos'

def _num(v, dec=0):
    return 'sin dato' if pd.isna(v) else f'{v:.{dec}f}'

def _pct(v):
    return 'sin dato' if pd.isna(v) else f'{v * 100:.0f}%'

def _cat(v, por_defecto='sin dato'):
    return por_defecto if pd.isna(v) else str(v)

def _si_no(v):
    return 'sin dato' if pd.isna(v) else ('sí' if v == 1 else 'no')

def _cuenta(v, singular, plural, sin_dato=None):
    """'1 crédito activo' / '2 créditos activos' / 'sin dato de créditos activos'."""
    if pd.isna(v):
        return sin_dato or f'sin dato de {plural}'
    n = int(v)
    return f'{n} {singular if n == 1 else plural}'


def redactar_ficha(f):
    partes = []

    partes.append(
        f'Perfil. Cliente de {_num(f.edad)} años, estrato {_num(f.estrato)}, '
        f'educación {_cat(f.nivel_educativo).lower()}, estado civil {_cat(f.estado_civil).lower()}, '
        f'{_cuenta(f.dependientes, "persona a cargo", "personas a cargo")}. '
        f'Vive en {_cat(f.ciudad)} (región {_cat(f.region).lower()}) hace '
        f'{_num(f.anios_ciudad_actual)} años, en vivienda {_cat(f.tipo_vivienda).lower()}.')

    partes.append(
        f'Ocupación. {_cat(f.descripcion_ocupacion, "ocupación no reportada").capitalize()}; '
        f'tipo de empleo {_cat(f.tipo_empleo).lower()}, profesión {_cat(f.profesion).lower()}. '
        f'Ingreso mensual {_millones(f.ingreso_mensual_cop)}, volatilidad del ingreso '
        f'{_num(f.volatilidad_ingreso, 2)}. Patrimonio declarado {_millones(f.patrimonio_cop)}.')

    partes.append(
        f'Situación crediticia. Score interno {_num(f.score_credito)}, '
        f'score de buró externo {_num(f.score_bureau_externo)}, '
        f'{_num(f.antiguedad_financiera)} años de historia en centrales. '
        f'{_cuenta(f.num_creditos_activos, "crédito activo", "créditos activos")} en '
        f'{_cuenta(f.num_entidades_reportadas, "entidad", "entidades", "un número de entidades sin dato")}, '
        f'{_cuenta(f.num_productos, "producto vigente", "productos vigentes")}. '
        f'Deuda total {_millones(f.deuda_total_actual)}, cuota mensual '
        f'{_millones(f.cuota_mensual_actual)}, relación deuda/ingreso '
        f'{_num(f.ratio_deuda_ingreso, 2)}. Utilización de tarjeta {_pct(f.utilizacion_tdc)}, '
        f'saldos de los últimos seis meses {_cat(f.tendencia_saldos, "sin tendencia clara")}. '
        f'Puede aportar garantía: {_si_no(f.tiene_garantia_disponible)}.')

    historia_mora = ('nunca registró mora' if pd.isna(f.meses_desde_ultima_mora)
                     else f'última mora hace {_num(f.meses_desde_ultima_mora)} meses')
    peor_mora = ('peor mora histórica sin dato' if pd.isna(f.max_dias_mora_historico)
                 else f'mora máxima histórica de {_num(f.max_dias_mora_historico)} días')
    partes.append(
        f'Comportamiento de pago. {_cuenta(f.moras_ultimos_24m, "mora", "moras")} en los '
        f'últimos 24 meses, {peor_mora}, {historia_mora}. '
        f'{_cuenta(f.consultas_centrales_6m, "consulta", "consultas")} a centrales de riesgo '
        f'en seis meses.')

    partes.append(
        f'Relación con la entidad. Vinculado hace {_num(f.anios_vinculado, 1)} años por canal '
        f'{_cat(f.origen_registro).lower()}; canal preferido {_cat(f.canal_preferido).lower()}, '
        f'uso digital alto: {_si_no(f.uso_digital_alto)}, '
        f'{_cuenta(f.transacciones_digitales_mes, "transacción digital", "transacciones digitales")} '
        f'al mes. Gasto mensual promedio {_millones(f.gasto_mensual_promedio)}, '
        f'de ese gasto {_pct(f.pct_gasto_esencial)} es esencial.')

    partes.append(
        f'Solicitud. Pide {_millones(f.monto_solicitado)} a {_num(f.plazo_solicitado)} meses '
        f'para {_cat(f.motivo_credito, "un motivo no declarado")}.')

    return ' '.join(partes)


# Redactamos sólo las fichas de los tres splits (~61k filas, ~2 min).
for nombre in partes:
    partes[nombre]['texto'] = partes[nombre].apply(redactar_ficha, axis=1)
print('Fichas redactadas.')

Fichas redactadas.


In [15]:
for i in [0, 1]:
    fila = partes['validation'].iloc[i]
    print('=' * 100)
    print(f'ETIQUETA: {"incumple en 12 meses" if fila.y else "paga al día"}')
    print('-' * 100)
    print(fila.texto)
    print()

ETIQUETA: paga al día
----------------------------------------------------------------------------------------------------
Perfil. Cliente de 41 años, estrato 4, educación universitaria, estado civil soltero, 2 personas a cargo. Vive en Bogotá (región andina) hace 10 años, en vivienda propia. Ocupación. Independiente con rut, docente; tipo de empleo independiente formal, profesión docente. Ingreso mensual 2.7 millones de pesos, volatilidad del ingreso 0.36. Patrimonio declarado sin dato. Situación crediticia. Score interno 850, score de buró externo 779, 21 años de historia en centrales. 0 créditos activos en 5 entidades, 1 producto vigente. Deuda total 0 pesos, cuota mensual 0 pesos, relación deuda/ingreso 0.00. Utilización de tarjeta 18%, saldos de los últimos seis meses estable. Puede aportar garantía: sí. Comportamiento de pago. 0 moras en los últimos 24 meses, mora máxima histórica de 0 días, nunca registró mora. 1 consulta a centrales de riesgo en seis meses. Relación con la enti

In [16]:
Path('data').mkdir(exist_ok=True)
for nombre, parte in partes.items():
    parte[['texto', 'y']].to_json(f'data/{nombre}.jsonl', orient='records',
                                  lines=True, force_ascii=False)
print('Escritos en data/: train.jsonl, validation.jsonl, test.jsonl')

Escritos en data/: train.jsonl, validation.jsonl, test.jsonl


---
# Parte B · Fine-tuning con LoRA

**Por qué la familia encoder.** La tarea es clasificar una secuencia completa: no se genera
texto, se lee la ficha entera y se emite una etiqueta. El objetivo de preentrenamiento de un
encoder —masked language modeling con atención bidireccional— construye una representación de
todo el documento a la vez, y aquí eso importa: la decisión sale de cruzar el score de buró que
está en el tercer bloque con el historial de mora que está en el cuarto. Un decoder causal sólo
mira hacia atrás y gastaría capacidad en generar; un encoder-decoder resolvería una
transformación entrada→salida que en esta tarea no existe.

In [17]:
from datasets import load_dataset

ds = load_dataset('json', data_files={
    'train':      'data/train.jsonl',
    'validation': 'data/validation.jsonl',
    'test':       'data/test.jsonl',
})
ds = ds.rename_columns({'texto': 'text', 'y': 'labels'})

for parte in ds:
    etiquetas = np.array(ds[parte]['labels'])
    print(f'{parte:<11} n={len(etiquetas):>6}  incumplen={etiquetas.sum():>5}  '
          f'tasa={etiquetas.mean():.4f}')

Generating train split: 21352 examples [00:00, 160484.80 examples/s]
Generating validation split: 20000 examples [00:00, 200266.14 examples/s]
Generating test split: 20000 examples [00:00, 146684.76 examples/s]


train       n= 21352  incumplen= 5338  tasa=0.2500
validation  n= 20000  incumplen=  729  tasa=0.0365
test        n= 20000  incumplen=  729  tasa=0.0365


## 9 · Baseline 1: la clase mayoritaria

El piso. Con 3,7% de incumplimiento, responder siempre "paga" acierta el 96,3% de las veces y
es **inútil**: no detecta ni un solo caso malo. Por eso *accuracy* no puede ser la métrica
principal de esta tarea.

In [18]:
from collections import Counter
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score)

y_val  = np.array(ds['validation']['labels'])
y_test = np.array(ds['test']['labels'])

clase_mayoritaria = Counter(ds['train']['labels']).most_common(1)[0][0]
pred_mayoritaria = np.full_like(y_test, clase_mayoritaria)

print(f'Clase mayoritaria = {clase_mayoritaria}')
print(f'  accuracy en test : {accuracy_score(y_test, pred_mayoritaria):.4f}')
print(f'  recall  en test  : {recall_score(y_test, pred_mayoritaria, zero_division=0):.4f}'
      '  <- no encuentra a nadie')
print(f'  PR-AUC (azar)    : {y_test.mean():.4f}  <- el piso real de la métrica principal')

Clase mayoritaria = 0
  accuracy en test : 0.9636
  recall  en test  : 0.0000  <- no encuentra a nadie
  PR-AUC (azar)    : 0.0365  <- el piso real de la métrica principal


### Por qué PR-AUC como métrica principal

Con 3,7% de positivos, *accuracy* premia al modelo que no hace nada y *ROC-AUC* se ve optimista
porque el eje de falsos positivos se diluye entre 19.000 negativos. La **precisión media
(PR-AUC)** mide lo único que le importa al negocio: de los clientes que el modelo señala como
riesgosos, cuántos lo son de verdad. Su piso no es 0,5 sino la prevalencia (0,037), así que
cualquier mejora es visible. Reportamos también ROC-AUC y F1 para dar contexto.

## 10 · Modelo base y tokenización del dominio

Antes de casarse con un modelo conviene mirar **cómo parte el vocabulario del dominio**. Un
modelo que rompe "desembolso" en cinco pedazos gasta longitud de contexto y aprende peor esa
palabra. Comparamos BETO (español) contra DistilBERT multilingüe sobre términos que aparecen
literalmente en nuestras fichas.

In [19]:
from transformers import AutoTokenizer

MODELO = 'dccuchile/bert-base-spanish-wwm-cased'
ALTERNATIVA = 'distilbert-base-multilingual-cased'

tok_es = AutoTokenizer.from_pretrained(MODELO)
tok_mx = AutoTokenizer.from_pretrained(ALTERNATIVA)

terminos = ['desembolso', 'incumplimiento', 'endeudamiento', 'hipotecario', 'centrales',
            'garantía', 'cuota', 'patrimonio', 'estrato', 'pensionado', 'mora']

print(f'{"término":<18}{"BETO":>6}{"mBERT":>8}')
for t in terminos:
    print(f'{t:<18}{len(tok_es.tokenize(t)):>6}{len(tok_mx.tokenize(t)):>8}')

ficha = ds['train'][0]['text']
print(f'\nFicha completa  BETO: {len(tok_es.tokenize(ficha))} tokens | '
      f'mBERT: {len(tok_mx.tokenize(ficha))} tokens')

c:\Users\camil\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\camil\.cache\huggingface\hub\models--dccuchile--bert-base-spanish-wwm-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


término             BETO   mBERT
desembolso             2       4
incumplimiento         1       4
endeudamiento          2       3
hipotecario            2       4
centrales              1       1
garantía               1       3
cuota                  1       2
patrimonio             1       1
estrato                2       2
pensionado             2       2
mora                   1       1

Ficha completa  BETO: 278 tokens | mBERT: 312 tokens


BETO fragmenta menos y deja la ficha más corta: cabe entera en la ventana y cada época cuesta
menos. Ese es el argumento técnico detrás de la elección, no "es el más popular".

In [20]:
from transformers import DataCollatorWithPadding

LONGITUD_MAX = 320

largos = [len(tok_es.tokenize(t)) for t in ds['train']['text'][:2000]]
print(f'tokens por ficha — mediana {np.median(largos):.0f}, p95 {np.percentile(largos, 95):.0f}, '
      f'max {np.max(largos)}')
print(f'fichas truncadas con LONGITUD_MAX={LONGITUD_MAX}: '
      f'{np.mean(np.array(largos) > LONGITUD_MAX) * 100:.1f}%')

def tokenizar(lote):
    return tok_es(lote['text'], truncation=True, max_length=LONGITUD_MAX)

ds_tok = ds.map(tokenizar, batched=True, remove_columns=['text'])
collator = DataCollatorWithPadding(tokenizer=tok_es)   # padding dinámico por lote
print(ds_tok)

tokens por ficha — mediana 276, p95 283, max 290
fichas truncadas con LONGITUD_MAX=320: 0.0%


Map: 100%|██████████| 20000/20000 [00:04<00:00, 4748.96 examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 21352
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20000
    })
})


## 11 · Baseline 2: TF-IDF + regresión logística

El baseline clásico de PLN. Si una bolsa de palabras ya resuelve la tarea, el transformer no se
justifica. Mide cuánta señal está en *qué palabras aparecen* y cuánta en *cómo se relacionan*.

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=60_000, sublinear_tf=True)
X_tr = vec.fit_transform(ds['train']['text'])
X_va = vec.transform(ds['validation']['text'])
X_te = vec.transform(ds['test']['text'])

logreg = LogisticRegression(max_iter=2000, C=1.0, random_state=SEMILLA)
logreg.fit(X_tr, ds['train']['labels'])

p_tfidf_val  = logreg.predict_proba(X_va)[:, 1]
p_tfidf_test = logreg.predict_proba(X_te)[:, 1]
print(f'TF-IDF + logreg | test  PR-AUC {average_precision_score(y_test, p_tfidf_test):.4f}  '
      f'ROC-AUC {roc_auc_score(y_test, p_tfidf_test):.4f}')

TF-IDF + logreg | test  PR-AUC 0.1383  ROC-AUC 0.8026


## 12 · Baseline 3: BETO **sin** fine-tuning

Este es el baseline que pide la asignación: *el mismo modelo base sin afinar*. Para un encoder,
"zero-shot" con una cabeza de clasificación recién inicializada es ruido puro y no informa nada.
La versión honesta es **congelar** el modelo y entrenar sólo una regresión logística sobre el
embedding `[CLS]`. Eso mide lo que BETO ya sabe de estas fichas antes de que LoRA toque nada, y
la diferencia contra el modelo afinado es el *delta real* del fine-tuning.

In [ ]:
from transformers import AutoModel
from torch.utils.data import DataLoader

base = AutoModel.from_pretrained(MODELO).to(device).eval()

@torch.no_grad()
def embeddings_cls(split, n=None):
    datos = ds_tok[split].select(range(min(n, len(ds_tok[split])))) if n else ds_tok[split]
    datos = datos.remove_columns([c for c in datos.column_names if c == 'labels'])
    salidas = []
    for lote in DataLoader(datos, batch_size=64, collate_fn=collator):
        lote = {k: v.to(device) for k, v in lote.items()}
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == 'cuda')):
            h = base(**lote).last_hidden_state[:, 0]      # token [CLS]
        salidas.append(h.float().cpu().numpy())
    return np.vstack(salidas)

N_SONDA = 8000   # submuestra para que la sonda no tarde
E_tr = embeddings_cls('train', N_SONDA)
E_te = embeddings_cls('test')

sonda = LogisticRegression(max_iter=3000, random_state=SEMILLA).fit(
    E_tr, np.array(ds['train']['labels'][:N_SONDA]))
p_base_test = sonda.predict_proba(E_te)[:, 1]
print(f'BETO congelado + sonda | test  PR-AUC {average_precision_score(y_test, p_base_test):.4f}  '
      f'ROC-AUC {roc_auc_score(y_test, p_base_test):.4f}')

del base; torch.cuda.empty_cache()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 15836.97it/s]
[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

## 13 · LoRA

En vez de mover los 110M de parámetros de BETO, congelamos el modelo y aprendemos dos matrices
de rango bajo por capa objetivo.

- **`r = 16`** (rango). Con `r=8` la tarea se queda corta: la señal está repartida en muchas
  pistas débiles (score, mora, utilización, tipo de empleo) y no en dos o tres palabras clave.
  `r=16` da capacidad suficiente sin pasar del 2% de parámetros entrenables; `r=32` no mejoró
  lo bastante para justificar el doble de cómputo.
- **`lora_alpha = 32`** — la regla usual `alpha ≈ 2·r`. El factor efectivo `alpha/r = 2`
  mantiene la magnitud del ajuste estable si luego cambiamos `r`.
- **`target_modules = ['query', 'value']`** — las proyecciones de atención de BERT. Es donde el
  modelo decide *a qué parte de la ficha mirar*, que es justo lo que hay que reaprender: BETO no
  sabe de fábrica que "mora máxima histórica" pesa más que "canal preferido". Dejamos `key`
  fuera (aporta poco sobre `query` y sube el costo) y las capas densas también.
- **`lora_dropout = 0.1`** — regularización barata sobre un train submuestreado.
- **`modules_to_save = ['classifier']`** — la cabeza de clasificación es nueva, no existe en el
  modelo preentrenado: hay que entrenarla completa, no adaptarla.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO, num_labels=2,
    id2label={0: 'paga', 1: 'incumple'}, label2id={'paga': 0, 'incumple': 1})

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['query', 'value'],
    modules_to_save=['classifier'],
)
modelo = get_peft_model(modelo, lora_cfg)
modelo.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

trainable params: 591,362 || all params: 110,443,780 || trainable%: 0.5354


## 14 · Entrenamiento

`Trainer` con lo mínimo. La métrica que decide el mejor checkpoint es PR-AUC sobre validación,
no accuracy. `learning_rate = 2e-4`: LoRA tolera un LR un orden por encima del full fine-tuning
porque sólo mueve las matrices de bajo rango.

In [ ]:
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    prob = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    pred = (prob >= 0.5).astype(int)
    return {
        'pr_auc':   average_precision_score(labels, prob),
        'roc_auc':  roc_auc_score(labels, prob),
        'f1':       f1_score(labels, pred, zero_division=0),
        'accuracy': accuracy_score(labels, pred),
    }

import inspect

deseados = dict(
    output_dir='./m1_lora_out',
    learning_rate=2e-4,                # LoRA tolera un LR alto: sólo mueve las matrices A y B
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='pr_auc',    # el mejor checkpoint lo decide PR-AUC, no accuracy
    greater_is_better=True,
    logging_steps=50,
    fp16=(device == 'cuda'),
    seed=SEMILLA,
    report_to='none',
)

# --- compatibilidad entre versiones de transformers -------------------------
admitidos = set(inspect.signature(TrainingArguments.__init__).parameters)

if 'eval_strategy' not in admitidos and 'evaluation_strategy' in admitidos:
    deseados['evaluation_strategy'] = deseados.pop('eval_strategy')   # < 4.41
if 'warmup_ratio' not in admitidos:
    # transformers 5.x eliminó warmup_ratio: warmup_steps acepta un float < 1
    # y significa exactamente lo mismo (proporción de los pasos totales).
    proporcion = deseados.pop('warmup_ratio')
    for alias in ('warmup_steps', 'warmup_step'):
        if alias in admitidos:
            deseados[alias] = proporcion
            break

usados = {k: v for k, v in deseados.items() if k in admitidos}

# Quedarse con el mejor checkpoint exige que sobrevivan las dos estrategias:
# si alguna no está soportada, se cae al último checkpoint en vez de romper.
if not ({'eval_strategy', 'evaluation_strategy'} & set(usados)) or 'save_strategy' not in usados:
    for k in ('load_best_model_at_end', 'metric_for_best_model', 'greater_is_better'):
        usados.pop(k, None)

omitidos = sorted(set(deseados) - set(usados))
if omitidos:
    print(f'transformers {transformers.__version__} no acepta estos argumentos; '
          f'quedan en su valor por defecto: {omitidos}\n')

args = TrainingArguments(**usados)
# ---------------------------------------------------------------------------

kwargs_trainer = dict(
    model=modelo,
    args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['validation'],
    data_collator=collator,
    compute_metrics=compute_metrics,
)
# `tokenizer=` se llama `processing_class=` desde transformers 5.x
admitidos_trainer = set(inspect.signature(Trainer.__init__).parameters)
kwargs_trainer['processing_class' if 'processing_class' in admitidos_trainer
               else 'tokenizer'] = tok_es

trainer = Trainer(**kwargs_trainer)
trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Pr Auc,Roc Auc,F1,Accuracy
1,0.434428,0.246436,0.147864,0.813738,0.231768,0.884650
2,0.426473,0.288167,0.155313,0.817976,0.238140,0.877150
3,0.425742,0.247974,0.159247,0.820438,0.245896,0.896650


TrainOutput(global_step=2004, training_loss=0.45022339854173793, metrics={'train_runtime': 1148.4544, 'train_samples_per_second': 55.776, 'train_steps_per_second': 1.745, 'total_flos': 9499943742930432.0, 'train_loss': 0.45022339854173793, 'epoch': 3.0})

## 15 · Umbral de decisión

El modelo entrena con 1 incumplimiento por cada 3 pagos, pero se aplica sobre una población con
3,7% de incumplimiento: la probabilidad que emite está corrida y 0,5 no es el corte correcto.
Elegimos el umbral que maximiza F1 **sobre validación** y lo congelamos antes de tocar el test.
Elegirlo sobre test sería hacer trampa.

In [ ]:
def probabilidades(split):
    logits = trainer.predict(ds_tok[split]).predictions
    return torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

p_lora_val  = probabilidades('validation')
p_lora_test = probabilidades('test')

rejilla = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_val, (p_lora_val >= u).astype(int), zero_division=0) for u in rejilla]
UMBRAL = float(rejilla[int(np.argmax(f1s))])
print(f'Umbral elegido en validación: {UMBRAL:.2f}  (F1 val = {max(f1s):.4f})')

Umbral elegido en validación: 0.65  (F1 val = 0.2517)


## 16 · Resultados

Todo sobre el **mismo conjunto de test**, que ningún modelo vio durante el entrenamiento ni
para elegir el umbral.

In [ ]:
def fila(nombre, prob, umbral=0.5):
    pred = (prob >= umbral).astype(int)
    return {
        'modelo': nombre,
        'PR-AUC':    round(average_precision_score(y_test, prob), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, prob), 4),
        'F1':        round(f1_score(y_test, pred, zero_division=0), 4),
        'Precisión': round(precision_score(y_test, pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, pred, zero_division=0), 4),
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
    }

tabla = pd.DataFrame([
    fila('Clase mayoritaria', np.full_like(y_test, y_test.mean(), dtype=float)),
    fila('TF-IDF + logreg', p_tfidf_test),
    fila('BETO sin fine-tuning (sonda [CLS])', p_base_test),
    fila('BETO + LoRA (afinado)', p_lora_test, UMBRAL),
])
display(tabla)

delta = average_precision_score(y_test, p_lora_test) - average_precision_score(y_test, p_base_test)
print(f'Delta PR-AUC del fine-tuning sobre el modelo base sin afinar: {delta:+.4f}')

,modelo,PR-AUC,ROC-AUC,F1,Precisión,Recall,Accuracy
0,Clase mayoritaria,0.0365,0.5000,0.0000,0.0000,0.0000,0.9636
1,TF-IDF + logreg,0.1383,0.8026,0.2218,0.1621,0.3512,0.9102
2,BETO sin fine-tuning (sonda [CLS]),0.1221,0.7740,0.1909,0.1390,0.3045,0.9059
3,BETO + LoRA (afinado),0.1723,0.8278,0.2646,0.2258,0.3196,0.9353


Delta PR-AUC del fine-tuning sobre el modelo base sin afinar: +0.0501


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred_lora = (p_lora_test >= UMBRAL).astype(int)
print('Matriz de confusión (filas = real, columnas = predicho):')
print(confusion_matrix(y_test, pred_lora))
print()
print(classification_report(y_test, pred_lora, target_names=['paga', 'incumple'], digits=4))

Matriz de confusión (filas = real, columnas = predicho):
[[18472   799]
 [  496   233]]

              precision    recall  f1-score   support

        paga     0.9739    0.9585    0.9661     19271
    incumple     0.2258    0.3196    0.2646       729

    accuracy                         0.9353     20000
   macro avg     0.5998    0.6391    0.6154     20000
weighted avg     0.9466    0.9353    0.9406     20000



## 17 · Ejemplos cualitativos

Los números no muestran qué aprendió el modelo. Aquí van cuatro fichas reales del test: dos que
acierta, una alarma falsa y una que se le escapa.

In [ ]:
orden = np.argsort(-p_lora_test)
idx_acierto        = [i for i in orden if y_test[i] == 1][:2]
idx_falso_positivo = [i for i in orden if y_test[i] == 0][:1]
idx_falso_negativo = [i for i in np.argsort(p_lora_test) if y_test[i] == 1][:1]

def mostrar(i, rotulo):
    print('=' * 100)
    print(f'{rotulo}  |  real = {"incumple" if y_test[i] else "paga"}  |  '
          f'p(incumple) = {p_lora_test[i]:.3f}  |  '
          f'predicho = {"incumple" if pred_lora[i] else "paga"}')
    print('-' * 100)
    print(ds['test'][int(i)]['text'])
    print()

for i in idx_acierto:
    mostrar(i, 'ACIERTO — detecta el riesgo')
mostrar(idx_falso_positivo[0], 'FALSO POSITIVO — alarma que no era')
mostrar(idx_falso_negativo[0], 'FALSO NEGATIVO — el que se le escapó')

ACIERTO — detecta el riesgo  |  real = incumple  |  p(incumple) = 0.823  |  predicho = incumple
----------------------------------------------------------------------------------------------------
Perfil. Cliente de 29 años, estrato 2, educación secundaria, estado civil unión libre, 2 personas a cargo. Vive en Ibagué (región andina) hace sin dato años, en vivienda familiar. Ocupación. Rebusque, agricultor; tipo de empleo informal, profesión agricultor. Ingreso mensual sin dato, volatilidad del ingreso 0.45. Patrimonio declarado 60.1 millones de pesos. Situación crediticia. Score interno 392, score de buró externo 513, 3 años de historia en centrales. 2 créditos activos en 5 entidades, 0 productos vigentes. Deuda total 6.4 millones de pesos, cuota mensual 225 mil pesos, relación deuda/ingreso 0.45. Utilización de tarjeta 52%, saldos de los últimos seis meses creciente. Puede aportar garantía: no. Comportamiento de pago. 3 moras en los últimos 24 meses, mora máxima histórica de 78 días, 

## 18 · Guardar el adaptador

LoRA guarda sólo las matrices pequeñas más la cabeza: unos pocos MB, no los 440 MB de BETO.

In [ ]:
modelo.save_pretrained('./m1_lora_adapter')
tok_es.save_pretrained('./m1_lora_adapter')
with open('./m1_lora_adapter/umbral.json', 'w', encoding='utf-8') as f:
    json.dump({'umbral': UMBRAL, 'metrica': 'f1 maximizado en validación'}, f, ensure_ascii=False)

peso = sum(os.path.getsize(os.path.join('./m1_lora_adapter', a))
           for a in os.listdir('./m1_lora_adapter')) / 1e6
print(f'Adaptador guardado en ./m1_lora_adapter ({peso:.1f} MB)')

Adaptador guardado en ./m1_lora_adapter (3.1 MB)


## 19 · Lectura honesta

Escriban aquí, con los números que acaban de salir arriba:

- ¿Cuánto subió el PR-AUC contra BETO sin afinar? Ese es el aporte del fine-tuning.
- **La vara real es TF-IDF.** Una bolsa de palabras ya multiplica por casi cuatro el piso de la
  prevalencia. Si BETO + LoRA no le gana con holgura, la mayor parte de la señal es léxica
  —"informal", "sin dato", "mora"— y el transformer aporta poco. Es un resultado válido y hay
  que decirlo, no esconderlo.
- ¿Dónde falla? Miren el falso negativo de la sección 17: ¿es un perfil con el score de buró
  ausente? ¿un informal con ingreso volátil?
- **Techo conocido.** Un *gradient boosting* sobre las columnas numéricas limpias —sin pasar
  por texto— alcanza ROC-AUC ≈ 0,835 y PR-AUC ≈ 0,188 en este mismo problema. El texto pierde
  precisión numérica al redondear 3.660.000 a "3,7 millones", y ese es el costo medido de
  tratar una tabla como lenguaje. Lo que se gana a cambio es un modelo que también podría leer
  una ficha escrita libremente por un asesor.
- **Sesgo declarado.** Aunque `sexo` se excluye, la ficha conserva ciudad, estrato, nivel
  educativo y tipo de vivienda. En Colombia el estrato está correlacionado con origen
  socioeconómico y racial: el modelo puede reproducir discriminación sin nombrarla nunca. Antes
  de cualquier uso real hay que auditar la tasa de falsos positivos por estrato y por región,
  no sólo la métrica global. Esta versión no lo hace, y ese es su límite más serio.